In [2]:
!pip install -q langgraph langchain-openai langchain-chroma langchain-huggingface sentence-transformers langchain-community

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 3.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.8/156.8 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.5/82.5 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.4/21.4 MB 116.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 55.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 473.0/473.0 kB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.0 MB/s eta 0:00:

In [6]:
import os
import getpass
from typing import List
from langchain_core.documents import Document
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# 1. 환경 설정
if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key를 입력하세요: ")

# 2. 임베딩 & DB 로드
print("--- 임베딩 모델 및 DB 로드 중... ---")
hf_embeddings = HuggingFaceEmbeddings(
    model_name="jhgan/ko-sroberta-multitask",
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

vectorstore = Chroma(
    persist_directory="/content/drive/MyDrive/kt_cs_agent/chroma_db_kt_terms",
    embedding_function=hf_embeddings,
    collection_name="kt_terms"
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# 3. 포맷팅 함수
def format_docs(docs: List[Document]):
    formatted_output = ""
    for i, doc in enumerate(docs):
        source = doc.metadata.get("source", "파일").split("/")[-1]
        page = doc.metadata.get("page", 0) + 1
        content = doc.page_content.replace('\n', ' ').strip()

        formatted_output += f"""
        📄 [문서 {i+1}] {source} (p.{page})
        ────────────────────────────────────────
        {content}
        ────────────────────────────────────────
        """
    return formatted_output

# ====================================================
# 🔥 핵심 로직: 키워드도 살리고, 검색도 하는 분기 처리
# ====================================================

llm = ChatOpenAI(model="gpt-5-nano", temperature=0)

# Step 1: 키워드 추출 프롬프트
keyword_prompt = ChatPromptTemplate.from_template(
    """
    당신은 상담 내용을 분석하여 검색 키워드를 추출하는 전문가입니다.
    [상담 요약]을 보고, 약관 데이터베이스 검색에 가장 적합한 '핵심 명사 키워드' 3~5개를 추출하세요.
    조사나 서술어는 제외하고, 띄어쓰기로 구분하여 출력하세요.
    오직 키워드만 공백으로 구분하여 출력하세요. (설명 금지)

    [예시]
    입력: 이사 가는데 인터넷 설치 비용이 드나요?
    출력: 인터넷 이전설치비 출동비 면제조건

    입력: 3년 약정인데 1년 쓰고 해지하면 얼마 내야 해요?
    출력: 중도해지 위약금 할인반환금 약정기간

    [상담 요약]
    {summary}
    """
)

# Step 2: 키워드 생성 체인 (여기까지는 문자열만 나옴)
keyword_chain = keyword_prompt | llm | StrOutputParser()

# Step 3: 최종 체인 (RunnableParallel 활용)
# 키워드 체인의 결과를 받아서 -> 두 갈래로 나눔
# 1) 'search_query': 키워드 그대로 통과 (RunnablePassthrough)
# 2) 'documents': 키워드로 검색 수행 후 포맷팅
final_chain = keyword_chain | {
    "search_query": RunnablePassthrough(),
    "documents": retriever | format_docs
}

# ====================================================
# 4. 실행 및 결과 확인
# ====================================================
summary_input = "고객이 현재 2년 약정으로 인터넷을 쓰고 있는데, 1년 만에 해지하면 위약금이 얼마나 나오는지 계산하는 공식을 궁금해하십니다."

print(f"\n🚀 입력된 상담 요약: {summary_input}")
print("=" * 60)

# 실행
result = final_chain.invoke({"summary": summary_input})

# 결과 출력 (이제 search_query 키를 사용할 수 있습니다!)
print(f"🗝️ [핵심] 추출된 키워드: {result['search_query']}")
print("-" * 60)
print(f"📄 [검색 결과] 근거 약관:\n{result['documents']}")
print("=" * 60)

--- 임베딩 모델 및 DB 로드 중... ---

🚀 입력된 상담 요약: 고객이 현재 2년 약정으로 인터넷을 쓰고 있는데, 1년 만에 해지하면 위약금이 얼마나 나오는지 계산하는 공식을 궁금해하십니다.
🗝️ [핵심] 추출된 키워드: 인터넷 약정 해지 위약금 계산식
------------------------------------------------------------
📄 [검색 결과] 근거 약관:

        📄 [문서 1] (이용약관전문)인터넷서비스이용약관_202509.pdf (p.147)
        ────────────────────────────────────────
        -  147 -    종료됩니다.  ※ 계약기간 내 인터넷 회선을 해지하는 경우 할인반환금 부과되며 금액은 각 할인상품의 할인반환금  산식을 따릅니다.   ※ 할인반환금 산식  1) GiGA WiFi home, KT WiFi 7D  - 2016년 4월~2023년 9월 8일 이전 가입한 인터넷 회선: ∑(월별할인액*월별할인반환금부과율)  - 2023년 9월 8일 이후 가입한 인터넷 회선: 할인 받은 총금액 X (1-이용기간에 따른 감면율)  2) 홈캠 안심 : 총 할인금액 X (1 – 사용기간할인율 / 약정기간할인율)  홈캠 안심 할인  이용고객이  사용기간을 케이티와  사전계약하고  계약기간 동안  소정의 요금을 할인   계약기간별 할인액 (VAT 포함)  구분 인터넷 구분 1년 2년  3년  기본카메라  인터넷 에센스 이상 3,300원 5,500원 6,600원  인터넷 에센스 미만 2,200원 4,400원 5,500원  추가카메라 2,200원 4,400원 5,500원    ※ 할인반환금  - 홈캠 안심 서비스를 약정기간 이내 해지 시 할인반환금이 부과됩니다.  - 할인반환금=이용개월수 *(사용기간 월이용료 - 계약기간별 월이용료)
        ────────────────────────────────────────